# Lenguajes y Autómatas II (SCD-1016)
## Sesión 21 · Fundamentos de Código de Tres Direcciones (TAC)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se convierte una expresión en una lista de instrucciones tan simples que cualquier procesador pueda ejecutarlas?
>
> **Objetivo de la sesión:** Entender qué es el código de tres direcciones y representar operaciones como cuádruplos, generando temporales únicos que no se pisen entre sí.

**Entregable:** tus funciones nuevo_temporal() y emitir_cuadruplo() en artesano_sesion21_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion21.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un contador y los campos de una instrucción; debes escribir el nombre del siguiente temporal o el cuádruplo completo en un buffer de texto. Las funciones no imprimen: escriben en dst, respetan la capacidad y dejan el contador actualizado.

Al final de la sesión tendrás: **Dos funciones en C que generan nombres de temporales únicos y formatean cuádruplos, verificadas por un harness con seis casos.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 21](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion21.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion21.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion21.h"

void nuevo_temporal(char *dst, int cap, int *contador);
void emitir_cuadruplo(char *dst, int cap, const char *op, const char *a1, const char *a2, const char *res);
```

- El contador se usa y después se incrementa: la primera llamada siempre produce _t0.
- Un argumento NULL se escribe como guion bajo en el cuádruplo; no se omite ni se inventa.
- Escribe siempre en el buffer recibido: no reserves memoria ni uses variables globales.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion21_c.c` junto con `artesano_sesion21_c.c`
> (o `ia_sesion21_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `nuevo_temporal(_t0)` | `_t0` |
| 2 | `nuevo_temporal(_t7)` | `_t7` |
| 3 | `cuadruplo de a + b` | `(+, a, b, _t0)` |
| 4 | `cuadruplo de _t0 * c` | `(*, _t0, c, _t1)` |
| 5 | `asignacion simple` | `(=, a, _, x)` |
| 6 | `goto sin operandos` | `(goto, L1, _, _)` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 21 - Cuadruplos y temporales unicos ===
Caso                               esperado:              obtenido:              resultado
[ 1] nuevo_temporal(_t0)                esperado: _t0          obtenido: _t0          OK
[ 2] nuevo_temporal(_t7)                esperado: _t7          obtenido: _t7          OK
[ 3] cuadruplo de a + b
     esperado: (+, a, b, _t0)
     obtenido: (+, a, b, _t0)
     -> OK
[ 4] cuadruplo de _t0 * c
     esperado: (*, _t0, c, _t1)
     obtenido: (*, _t0, c, _t1)
     -> OK
[ 5] asignacion simple                  esperado: (=, a, _, x) obtenido: (=, a, _, x) OK
[ 6] goto sin operandos
     esperado: (goto, L1, _, _)
     obtenido: (goto, L1, _, _)
     -> OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion21.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] nuevo_temporal() produce _t0, _t7 y _t10 a partir del contador y lo deja incrementado.
- [ ] emitir_cuadruplo() respeta el formato (op, a1, a2, res) incluso cuando faltan argumentos.
- [ ] Los seis casos del harness salen OK con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion21_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion21.h
/* api_sesion21.h - contrato de la Sesion 21
 * Cuádruplos y temporales únicos
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION21_H
#define API_SESION21_H

void nuevo_temporal(char *dst, int cap, int *contador);
void emitir_cuadruplo(char *dst, int cap, const char *op, const char *a1, const char *a2, const char *res);

#endif /* API_SESION21_H */


In [ ]:
%%writefile artesano_sesion21_c.c
/* artesano_sesion21_c.c - Fase 1: El Artesano (sin IA)
 * Cuádruplos y temporales únicos.
 *
 * Implementa lo que declara api_sesion21.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion21.h"
#include <stdio.h>
#include <string.h>

void nuevo_temporal(char *dst, int cap, int *contador) {
          (void)dst; (void)cap; (void)contador; /* TODO: escribe _tK y avanza el contador */
      }
      
      void emitir_cuadruplo(char *dst, int cap, const char *op, const char *a1, const char *a2, const char *res) {
          (void)dst; (void)cap; (void)op; (void)a1; (void)a2; (void)res; /* TODO: formatea el cuadruplo */
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion21_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion21_c.c
/* ia_sesion21_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion21.h. No puede traer main().
 */
#include "api_sesion21.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion21_c.c
/* harness_sesion21_c.c - auditoria automatica de la Sesion 21
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion21_c.c artesano_sesion21_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion21.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 21 - Cuadruplos y temporales unicos ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int c = 0; char b[32] = ""; nuevo_temporal(b, 32, &c); verificar_str("nuevo_temporal(_t0)", "_t0", b); }
    { int c = 7; char b[32] = ""; nuevo_temporal(b, 32, &c); verificar_str("nuevo_temporal(_t7)", "_t7", b); }
    { char b[64] = ""; emitir_cuadruplo(b, 64, "+", "a", "b", "_t0"); verificar_str_largo("cuadruplo de a + b", "(+, a, b, _t0)", b); }
    { char b[64] = ""; emitir_cuadruplo(b, 64, "*", "_t0", "c", "_t1"); verificar_str_largo("cuadruplo de _t0 * c", "(*, _t0, c, _t1)", b); }
    { char b[64] = ""; emitir_cuadruplo(b, 64, "=", "a", NULL, "x"); verificar_str("asignacion simple", "(=, a, _, x)", b); }
    { char b[64] = ""; emitir_cuadruplo(b, 64, "goto", "L1", NULL, NULL); verificar_str_largo("goto sin operandos", "(goto, L1, _, _)", b); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion21.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion21_c.c artesano_sesion21_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion21_c.c ia_sesion21_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion21.md` a `auditorias/auditoria_sesion21.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
